In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

username = "aacuser"
password = "SNHU1234"

db = AnimalShelter(username, password)

df = pd.DataFrame.from_records(db.read({}))
df.drop(columns=['_id'], inplace=True)

#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Logo + encoding
image_filename = 'Grazioso Salvare Logo.png'
with open(image_filename, 'rb') as f:
    encoded_image = base64.b64encode(f.read()).decode()

app.layout = html.Div([

    # Branding
    html.A(
        html.Img(
            src="data:image/png;base64,{}".format(encoded_image),
            style={'height': '80px'}
        ),
        href="https://www.snhu.edu",
        target="_blank"
    ),

    html.H3("Dashboard Developed by Alexander Berthiaume"),

    html.Center(html.B(html.H1('CS-340 Dashboard'))),
    html.Hr(),

    # Interactive filter controls
    html.Div([
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'WATER'},
                {'label': 'Mountain / Wilderness Rescue', 'value': 'MOUNTAIN'},
                {'label': 'Disaster / Individual Tracking', 'value': 'DISASTER'},
                {'label': 'Reset', 'value': 'RESET'}
            ],
            value='RESET',
            labelStyle={'display': 'inline-block'}
        )
    ]),

    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True}
                 for i in df.columns],
        data=df.to_dict('records'),

        page_size=10,
        sort_action='native',
        filter_action='native',
        row_selectable='single',
        selected_rows=[],
        style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'padding': '5px'}
    ),

    html.Br(),
    html.Hr(),

    html.Div(className='row', style={'display': 'flex'}, children=[

        html.Div(id='graph-id', className='col s12 m6'),

        html.Div(id='map-id', className='col s12 m6')
    ])
])

#############################################
# Controller (Callbacks)
#############################################

# MongoDB filtering logic
@app.callback(
    Output('datatable-id', 'data'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):

    query = {}

    if filter_type == "WATER":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix",
                              "Chesapeake Bay Retriever",
                              "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "MOUNTAIN":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd",
                              "Alaskan Malamute",
                              "Old English Sheepdog",
                              "Siberian Husky",
                              "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "DISASTER":
        query = {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher",
                              "German Shepherd",
                              "Golden Retriever",
                              "Bloodhound",
                              "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    elif filter_type == "RESET":
        query = {}

    df_filtered = pd.DataFrame.from_records(db.read(query))
    df_filtered.drop(columns=['_id'], inplace=True)

    return df_filtered.to_dict('records')


# Pie chart (required second chart)
@app.callback(
    Output('graph-id', "children"),
    Input('datatable-id', "derived_virtual_data")
)
def update_graphs(viewData):

    if viewData is None:
        return []

    dff = pd.DataFrame(viewData)

    fig = px.pie(
        dff,
        names='breed',
        title='Breed Distribution by Rescue Type'
    )

    return [dcc.Graph(figure=fig)]


# Style highlight
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    Input('datatable-id', 'selected_columns')
)
def update_styles(selected_columns):

    if not selected_columns:
        return []

    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Safe map callback
@app.callback(
    Output('map-id', "children"),
    [
        Input('datatable-id', "derived_virtual_data"),
        Input('datatable-id', "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):

    if not viewData:
        return []

    dff = pd.DataFrame(viewData)

    row = 0 if not index else index[0]

    lat = dff.iloc[row]['location_lat']
    lon = dff.iloc[row]['location_long']

    return [
        dl.Map(
            style={'width': '1000px', 'height': '500px'},
            center=[lat, lon],
            zoom=10,
            children=[
                dl.TileLayer(),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(dff.iloc[row]['breed']),
                        dl.Popup([
                            html.H4("Animal Name"),
                            html.P(dff.iloc[row]['name'])
                        ])
                    ]
                )
            ]
        )
    ]


# RUN APP
app.run_server(debug=True)

Dash app running on https://plutofreddie-oreganogranite-3000.codio.io/proxy/8050/
